# KT Stories — Voice Story Generator

Generates a narrated story MP3 + YouTube MP4 using local voice cloning (XTTS v2).

**Runtime:** `Runtime → Change runtime type → T4 GPU` before running.

---
### Workflow
1. **Setup** — install deps, clone repo
2. **Mount Drive** — connect your Google Drive for assets & output
3. **Configure** — pick script, voice, language, image duration
4. **Run** — generates MP3 + MP4
5. **Download** — files saved to Drive and downloadable directly

## Step 1 — Install dependencies

In [ ]:
# Check GPU
!nvidia-smi

# Install PyTorch (Colab usually has it, but ensure CUDA version matches)
import torch
print(f"PyTorch: {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")

# Install project dependencies
!pip install -q coqui-tts pydub soundfile

# ffmpeg is pre-installed on Colab; verify
!ffmpeg -version | head -1

## Step 2 — Clone the repo

In [ ]:
import os

REPO_DIR = "/content/kt-stories"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/avanikutle/kt-stories.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
!ls -la

## Step 3 — Mount Google Drive (recommended)

Store your assets (scripts, voices, audio, images) in Drive so they persist across sessions.

**Suggested Drive folder structure:**
```
My Drive/
└── KT-Stories/
    ├── script/      ← your .txt scripts
    ├── Voice/       ← male_voice.mp3 / female_voice.mp4
    ├── audio/       ← start.mp3, end.mp3, background.mp3
    └── image/       ← PNG/JPG images for slideshow
```

If you skip this step, the repo's built-in sample files are used.

In [ ]:
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
DRIVE_FOLDER = "KT-Stories"  # @param {type:"string"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_BASE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"

    import shutil, pathlib

    def sync_from_drive(folder_name):
        src = pathlib.Path(DRIVE_BASE) / folder_name
        dst = pathlib.Path(REPO_DIR) / folder_name
        if src.exists():
            dst.mkdir(exist_ok=True)
            for f in src.iterdir():
                shutil.copy2(str(f), str(dst / f.name))
            print(f"  Synced {folder_name}/ from Drive ({len(list(src.iterdir()))} files)")
        else:
            print(f"  {folder_name}/ not found in Drive — using repo defaults")

    for folder in ["script", "Voice", "audio", "image"]:
        sync_from_drive(folder)
else:
    print("Using built-in sample files from the repo.")

## Step 4 — Configuration

Set your options below, then run the cell.

In [ ]:
import pathlib

# ── Show available scripts ───────────────────────────────────────────────────
script_dir = pathlib.Path(REPO_DIR) / "script"
scripts = sorted(script_dir.glob("*.txt"))
print("Available scripts:")
for i, s in enumerate(scripts):
    print(f"  {i}: {s.name}")

# ── Configuration form ───────────────────────────────────────────────────────
SCRIPT_INDEX  = 1          # @param {type:"integer"}  — index from list above
LANGUAGE      = "te"       # @param ["en", "te", "hi", "es", "fr", "de", "ar"]
VOICE_MODE    = "male"     # @param ["male", "female", "both"]
IMAGE_DURATION = 15        # @param [15, 20]
OUTPUT_NAME   = ""         # @param {type:"string"}  — leave blank to use script filename

# ── Resolve config ───────────────────────────────────────────────────────────
selected_script = scripts[SCRIPT_INDEX]
output_name = OUTPUT_NAME.strip() or selected_script.stem

print("\n─── Configuration ───────────────────────────────")
print(f"  Script    : {selected_script.name}")
print(f"  Language  : {LANGUAGE}")
print(f"  Voice     : {VOICE_MODE}")
print(f"  Image dur : {IMAGE_DURATION}s")
print(f"  Output    : output/{output_name}/")

## Step 5 — Generate voice narration

⏱ First run downloads the XTTS v2 model (~1.8 GB). Subsequent runs use the cache.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from voice_engine import VoiceEngine

base = pathlib.Path(REPO_DIR)
out_dir = base / "output" / output_name
out_dir.mkdir(parents=True, exist_ok=True)

engine = VoiceEngine(base)
segments = engine.generate(
    script_path=selected_script,
    language=LANGUAGE,
    voice_mode=VOICE_MODE,
    output_dir=out_dir,
)
print(f"\nGenerated {len(segments)} segment(s).")

## Step 6 — Mix audio (narration + background + ducking)

In [ ]:
from audio_mixer import AudioMixer

mixer = AudioMixer(base)
mp3_path = mixer.mix(
    segments=segments,
    output_dir=out_dir,
    output_name=output_name,
)
print(f"MP3 ready: {mp3_path}")

## Step 7 — Build video (image slideshow + audio)

In [ ]:
from video_builder import VideoBuilder

builder = VideoBuilder(base)
mp4_path = builder.build(
    audio_path=mp3_path,
    output_dir=out_dir,
    output_name=output_name,
    image_duration=IMAGE_DURATION,
)
print(f"MP4 ready: {mp4_path}")

## Step 8 — Download output files

In [ ]:
from google.colab import files

print("Downloading MP3 (Spotify)…")
files.download(str(mp3_path))

print("Downloading MP4 (YouTube)…")
files.download(str(mp4_path))

## (Optional) Save output to Google Drive

In [ ]:
import shutil

if USE_GOOGLE_DRIVE:
    drive_out = pathlib.Path(DRIVE_BASE) / "output" / output_name
    drive_out.mkdir(parents=True, exist_ok=True)

    shutil.copy2(str(mp3_path), str(drive_out / mp3_path.name))
    shutil.copy2(str(mp4_path), str(drive_out / mp4_path.name))

    print(f"Saved to Drive: MyDrive/{DRIVE_FOLDER}/output/{output_name}/")
    print(f"  {mp3_path.name}")
    print(f"  {mp4_path.name}")
else:
    print("Drive not mounted — use the download cell above.")